In [2]:
import fs from 'node:fs';
import decompress from "npm:decompress";

type FileResult =
  | { success: true; filePath: string }
  | { success: false; error: 'NOT_FOUND' | 'PERMISSION_DENIED' };
function getChatGptZip(): FileResult {
  const directory = './data/chatgpt';

  const files = fs.readdirSync(directory);
  const matches = files.filter(file => file.endsWith(".zip"));
  if (matches.length == 0) {
    return {
      sucess: false,
      error: "NOT_FOUND"
    }
  }
  return {
    success: true,
    filePath: './data/chatgpt/' + matches[0]
  }
}

function clearDirIfExists(dir: string) {
  if (fs.existsSync(dir)) {
    // recursive: true deletes subfolders
    // force: true ignores the error if the folder doesn't exist
    console.log("Cleared contents of ", dir)
    fs.rmSync(dir, { recursive: true, force: true });
  }
}

const chatgptZip: FileResult = getChatGptZip()
const targetDir = "./data/chatgpt/output"
if (!chatgptZip.success) {
  console.warn("Couldn't find chatgpt zip")
  exit(1)
}
console.log("Unzipping file : ", chatgptZip.filePath)
clearDirIfExists(targetDir)
await decompress(chatgptZip.filePath, targetDir, {
  filter: file => file.path == "conversations.json"
});


Unzipping file :  ./data/chatgpt/63052a98b0d2d280658c12257a596058aff1a44e95e141bc2a5717eae6f3d426-2026-01-06-20-23-04-35a67a168c55430e97c25247d68e588c.zip
Cleared contents of  ./data/chatgpt/output


[
  {
    mode: 384,
    mtime: 2026-01-06T20:23:04.000Z,
    path: "conversations.json",
    type: "file",
    data: <Buffer 5b 7b 22 74 69 74 6c 65 22 3a 20 22 44 6f 63 6b 65 72 66 69 6c 65 20 66 6f 72 20 45 6c 65 63 74 72 6f 6e 20 41 70 70 22 2c 20 22 63 72 65 61 74 65 5f ... 42707499 more bytes>
  }
]

In [3]:
import pl from 'npm:nodejs-polars';
import { readFile } from 'node:fs/promises';
import { display } from "https://deno.land/x/display@v1.1.2/mod.ts";


// about 50mb
const jsonPath = "./data/chatgpt/output/conversations.json"

// Reading
const rawJsonData = await readFile(jsonPath, 'utf-8');
const chatgptRawData = JSON.parse(rawJsonData);
// use snake_case here cus thats what the json uses + everyone else in data too
interface flattenedChatgptMessage {
  conversation_id: string,
  conversation_title: string,
  message_id: string,
  parent: string,
  children: string[],
  create_time: string,
  role: "user" | "assistant",
  content_type: string,
  content_parts: string[],
}

function extractMessages(conversationJson: any): flattenedChatgptMessage[] {
  const conversation_id = conversationJson.id
  const conversation_title = conversationJson.title
  const chatMessagesObjects = Object.values(conversationJson.mapping)
  const messages = []
  chatMessagesObjects.forEach(chatMessage => {
    try {
      const flattenedMessage: flattenedChatgptMessage = {
        conversation_id: conversation_id ?? "",
        conversation_title: conversation_title?? "",
        message_id: chatMessage.id ?? "",
        parent: chatMessage.parent ?? "",
        children: chatMessage.children ?? [],
        create_time: chatMessage.message?.create_time ?? "",
        role: chatMessage.message?.author.role ?? "",
        content_type: chatMessage.message?.content.content_type ?? "",
        content_parts: ""
      }
      if (flattenedMessage.content_type === "text") {
        flattenedMessage.content_parts = chatMessage.message?.content.parts ?? []
      }
      messages.push(flattenedMessage)
    } catch {
      console.error("Error encountered whilst trying to process message \n", chatMessage)
    }
  })

  return messages
}

function displayTable(df: pl.DataFrame, limit:number = 10) {
  console.log(df.shape)
  console.table(df.head(limit).toRecords());
}

const flattenedData = chatgptRawData.map(extractMessages).flat()
let df = pl.DataFrame(flattenedData)

df = df.withColumn(
    pl
    .col("create_time")
    .cast(pl.Float64)
    .mul(1000)
    .cast(pl.Datetime("ms"))        // cast to Polars datetime
    .alias("datetime")
  )
  

// we have sent over 18,000 messages :o
displayTable(df)

{ height: 18452, width: 10 }
┌───────┬────────────────────────────────────────┬───────────────────────────────┬────────────────────────────────────────┬────────────────────────────────────────┬────────────────────────────────────────────┬──────────────────────┬─────────────┬─────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
const transformedDf = df
  .select("conversation_title", "role", "content_parts")
  .where(pl.col("role").eq(pl.lit("assistant")))

const languages = [
  "html",
  "js",
  "css",
  "scss",
  "ts",
  "jsx",
  "tsx",
  "sql",
  "bash",
  "sh",
  "python",
  "java",
  "javascript",
  "typescript",
  "markdown",
  "md",
  "mermaid",
  "rust",
  ""
]
const udfedRecords = df.map((row) => {
  const obj: Record<string, any> = Object.fromEntries(df.columns.map((col, i) => [col, row[i]])); // nodejs-polars is in caveman time, we must hit rock together
  languages.forEach((lang) => {
    obj[`${lang}_count`] = String(obj["content_parts"]).split("```" + lang + "\n").length - 1
  })
  return obj
})


var anotherDf = pl.DataFrame(udfedRecords)
var langAggDf = anotherDf

const colsToSum = languages.map(lang => {
  return `${lang}_count`
})

const dfWithSum = anotherDf.select(colsToSum).sum().unpivot( // tf happended to melt!!!
    [], 
    colsToSum,
    {
        variableName: 'language', 
        valueName: 'count'        
    }
).sort('count', true).withColumn(
    pl.col('language').str.replace('_count', '')
);


displayTable(dfWithSum, 20)
// python on top, but js really if you count javascript + typescript. 
// 3000 python code blocks is a bit nuts, 

{ height: 19, width: 2 }
┌───────┬──────────────┬───────┐
│ (idx) │ language     │ count │
├───────┼──────────────┼───────┤
│     0 │ "python"     │  3474 │
│     1 │ "bash"       │  2336 │
│     2 │ "javascript" │  1082 │
│     3 │ "ts"         │   623 │
│     4 │ "html"       │   516 │
│     5 │ "sh"         │   505 │
│     6 │ "jsx"        │   448 │
│     7 │ "tsx"        │   447 │
│     8 │ "js"         │   441 │
│     9 │ "css"        │   379 │
│    10 │ "sql"        │   218 │
│    11 │ "typescript" │   211 │
│    12 │ "markdown"   │    47 │
│    13 │ "mermaid"    │    30 │
│    14 │ "scss"       │    16 │
│    15 │ "java"       │    13 │
│    16 │ "md"         │     9 │
│    17 │ "rust"       │     4 │
│    18 │ "ipynb"      │     0 │
└───────┴──────────────┴───────┘


In [40]:

// Example DataFrame
var df3 = pl.DataFrame({
  a: [1, 2, 3],
  b: [10, 20, 30]
});

// Apply function on a single column
df3 = df3.mapElements((row) => {
  row["yo"] = "yo"
  return row
})
console.log(df3)

TypeError: df3.mapElements is not a function

In [ ]:
// I have asked 6918 message.
const transformedDf = df
  .select("conversation_title", "role", "content_parts")
  .where(pl.col("role").eq(pl.lit("user")))
  .withRowIndex()
  .where(pl.col("index").notEquals(pl.lit(5))) // 
  .where(pl.col("index").notEquals(pl.lit(3))) 

displayTable(transformedDf, 10)

{ height: 6916, width: 4 }
┌───────┬───────┬───────────────────────────────┬────────┬──────────────────────────────────────────────────────────────────────────────────────────┐
│ (idx) │ index │ conversation_title            │ role   │ content_parts                                                                            │
├───────┼───────┼───────────────────────────────┼────────┼──────────────────────────────────────────────────────────────────────────────────────────┤
│     0 │     0 │ "Dockerfile for Electron App" │ "user" │ ""                                                                                       │
│     1 │     1 │ "Dockerfile for Electron App" │ "user" │ "should i include a docker file for my electron typescript react app?"                   │
│     2 │     2 │ "Dockerfile for Electron App" │ "user" │ "is zsh the default for mac?"                                                            │
│     3 │     4 │ "Dockerfile for Electron App" │ "user" │ "can i build g

In [ ]:
// chat has given  asked 7373 message. alot
const transformedDf = df
  .select("conversation_title", "role", "content_parts")
  .withColumns(
    pl.col("content_parts").str.slice(0, 30) /// shhhh
  )
  .where(pl.col("role").eq(pl.lit("assistant")))
displayTable(transformedDf, 15)

{ height: 7373, width: 3 }
┌───────┬───────────────────────────────┬─────────────┬────────────────────────────────────┐
│ (idx) │ conversation_title            │ role        │ content_parts                      │
├───────┼───────────────────────────────┼─────────────┼────────────────────────────────────┤
│     0 │ "Dockerfile for Electron App" │ "assistant" │ ""                                 │
│     1 │ "Dockerfile for Electron App" │ "assistant" │ "Short answer: **usually no** —"   │
│     2 │ "Dockerfile for Electron App" │ "assistant" │ "Yes — **zsh is the default she"   │
│     3 │ "Dockerfile for Electron App" │ "assistant" │ "This is the **classic Electron"   │
│     4 │ "Dockerfile for Electron App" │ "assistant" │ "Yes — you **can** build graphs"   │
│     5 │ "Dockerfile for Electron App" │ "assistant" │ "Here’s a **clean, conventional"   │
│     6 │ "Dockerfile for Electron App" │ "assistant" │ "If you mean **`ll` in the term"   │
│     7 │ "Dockerfile for Electron App" │ "

In [ ]:
// 1000 unique chats, bruh
displayTable(df.select("conversation_id").unique("conversation_id"), 15)

{ height: 1478, width: 1 }
┌───────┬────────────────────────────────────────┐
│ (idx) │ conversation_id                        │
├───────┼────────────────────────────────────────┤
│     0 │ "68ae1b1d-17ec-8329-b772-00cd58d568d8" │
│     1 │ "66e34ad6-823c-800b-8030-09d51712c7ae" │
│     2 │ "62f4c378-b3bc-4773-956d-584a6d52035a" │
│     3 │ "bb1f6a13-b7b0-4ff0-8906-74df9c14037d" │
│     4 │ "678fd545-9cd8-800b-bee2-9893363de7cf" │
│     5 │ "6810bdd3-af80-800b-8d4a-a0c0d3b5323b" │
│     6 │ "68178a1d-1d74-800b-9293-9b7462367714" │
│     7 │ "34c92c5e-4b6c-47bb-a60c-ea56bb00ea7c" │
│     8 │ "6846d656-e634-800b-862a-efbdc8a9da39" │
│     9 │ "67649e87-a4f8-800b-8f13-1a968440d3c1" │
│    10 │ "68ce5b23-6450-832d-8ab5-aba068b47969" │
│    11 │ "68f67e39-2d38-832d-a824-0c7262ac712a" │
│    12 │ "68ad7e32-6cc0-8324-a08d-7129621c9664" │
│    13 │ "680fdee9-4e68-800b-83ac-b7b11e76f59d" │
│    14 │ "681cdbdc-66bc-800b-972f-aad446c951f4" │
└───────┴────────────────────────────────────────┘


In [ ]:
// 4 chats per convo wo
const transformedDf = df
  .groupBy("conversation_id")
  .agg(
    (pl.col("role").eq(pl.lit("user"))).sum().alias("user_chats")
  )

const stats =  transformedDf.select([
  pl.col("user_chats").max().alias("max_chats_in_a_convo"),
  pl.col("user_chats").sum().alias("total_chats"),
  pl.col("user_chats").mean().alias("avg_chats_per_convo")
]);
displayTable(stats)

{ height: 1, width: 3 }
┌───────┬──────────────────────┬─────────────┬─────────────────────┐
│ (idx) │ max_chats_in_a_convo │ total_chats │ avg_chats_per_convo │
├───────┼──────────────────────┼─────────────┼─────────────────────┤
│     0 │                   80 │        6918 │ 4.680649526387009   │
└───────┴──────────────────────┴─────────────┴─────────────────────┘


In [ ]:
// There are 10 different content_types, i only process text :P
displayTable(df.select("content_type").unique("content_type"), 15)

{ height: 11, width: 1 }
┌───────┬───────────────────────────┐
│ (idx) │ content_type              │
├───────┼───────────────────────────┤
│     0 │ "system_error"            │
│     1 │ "text"                    │
│     2 │ "user_editable_context"   │
│     3 │ "tether_browsing_display" │
│     4 │ "reasoning_recap"         │
│     5 │ "code"                    │
│     6 │ ""                        │
│     7 │ "multimodal_text"         │
│     8 │ "tether_quote"            │
│     9 │ "thoughts"                │
│    10 │ "execution_output"        │
└───────┴───────────────────────────┘


In [ ]:
// 5 different roles woaw
displayTable(df.select("role").unique("role"), 15)

{ height: 5, width: 1 }
┌───────┬─────────────┐
│ (idx) │ role        │
├───────┼─────────────┤
│     0 │ "assistant" │
│     1 │ "user"      │
│     2 │ ""          │
│     3 │ "system"    │
│     4 │ "tool"      │
└───────┴─────────────┘
